<a href="https://colab.research.google.com/github/Zattyla/K-UVR-Colab/blob/main/K-UVR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 K-UVR | Audio Separator (v1.0)
*Advanced Vocal & Instrumental Extraction Pipeline*

### 🚀 How to use:
1.  **Select GPU:** Ensure you are using a **T4 GPU** (`Runtime` -> `Change runtime type`).
2.  **Run Cell 1:** Install the core engine and dependencies.
3.  **Run Cell 2:** Download the professional model library (Kim, MDX-Net, etc.).
4.  **Upload Audio:** Click the **Folder icon** on the left and upload your song to the `inputs` folder.
5.  **Run Cell 3:** Enter your filename, choose a model, and toggle **DEREVERB** for cleaner AI covers.

### 🔬 Model Guide:
- **Kim_Vocal_2:** Your "Go-to" model. Incredible vocal isolation.
- **MDX_UVR_Vocal_FT:** Use this if the song has heavy backing vocals or complex harmonies.
- **UVR_Inst_HQ_3:** Use this to get a studio-quality instrumental track for karaoke or mixing.
- **De-reverb:** Automatically cleans the "hall effect" or echo from the vocals.

---

In [ ]:
# @title 1. Workspace Initialization
import os
from google.colab import drive

# Create folders for the pipeline
folders = ["inputs", "outputs", "models"]
for folder in folders:
    os.makedirs(f"/content/{folder}", exist_ok=True)

print("✅ Workspace folders created: /inputs, /outputs, /models")

In [ ]:
# @title 2. Install UVR Engine
print("⏳ Installing dependencies... (This may take 1-2 minutes)")
!pip install -U numpy>=2.0.0
!pip install -q audio-separator[gpu]
print("✅ Installation complete! UVR Engine is ready.")

In [ ]:
# @title 3. Download Model Library
import os
import requests

# Professional Model Mapping (Direct Links)
models = {
    "Kim_Vocal_2": "https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/Kim_Vocal_2.onnx",
    "MDX_UVR_Vocal_FT": "https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/UVR-MDX-NET-Voc_FT.onnx",
    "UVR_Inst_HQ_3": "https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/UVR-MDX-NET-Inst_HQ_3.onnx",
    "Reverb_HQ": "https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/UVR-DeReverb-HQ.onnx"
}

print("⏬ Downloading professional models... (This will take a few minutes)")

for name, url in models.items():
    file_name = url.split('/')[-1]
    model_path = f"/content/models/{file_name}"

    if not os.path.exists(model_path):
        print(f"Downloading {name}...")
        # Using !wget because it shows a real progress bar in Colab
        !wget -q --show-progress -O "{model_path}" "{url}"
    else:
        print(f"✅ {name} already exists. Skipping.")

print("\n✨ All models are ready! Check the '/models' folder to confirm.")

In [ ]:
# @title 4. Run Audio Separation
import os

# @markdown ### 📂 File Settings
# @markdown Enter the filename of the song you uploaded to the 'inputs' folder:
FILE_NAME = "song.mp3" # @param {type:"string"}

# @markdown ### 📝 Model Strategy
# @markdown Choose the best model for your specific song:
MODEL_CHOICE = "Kim_Vocal_2" # @param ["Kim_Vocal_2", "MDX_UVR_Vocal_FT", "UVR_Inst_HQ_3"]

# @markdown ### 🧹 Post-Processing
# @markdown Remove echo/reverb from the extracted vocals? (Highly recommended for RVC)
DEREVERB = True # @param {type:"boolean"}

input_path = f"/content/inputs/{FILE_NAME}"
model_file = models[MODEL_CHOICE]

if os.path.exists(input_path):
    print(f"🚀 Starting separation: {FILE_NAME} using {MODEL_CHOICE}...")

    # Execution Command
    !python -m audio_separator.utils.cli "{input_path}" \
    --model_filename "{model_file}" \
    --output_dir "/content/outputs" \
    --output_format WAV \
    --normalization 0.9

    if DEREVERB:
        print("🧹 Cleaning vocals (Removing Reverb)...")
        # Identifies the generated vocal file to clean it
        vocal_files = [f for f in os.listdir("/content/outputs") if "Vocals" in f and f.endswith(".wav")]
        if vocal_files:
            latest_vocal = os.path.join("/content/outputs", vocal_files[-1])
            !audio-separator "{latest_vocal}" \
                --model_filename "UVR-DeReverb-HQ.onnx" \
                --output_dir "/content/outputs" \
                --output_format WAV
            print("✅ Vocals are now dry and clean!")

    print("✨ Process finished! Check the '/outputs' folder.")
else:
    print(f"❌ Error: File '{FILE_NAME}' not found in /content/inputs. Please upload it first.")

In [ ]:
# @title 5. Cleanup & GPU Refresh
import torch, gc

# Clear GPU cache to prevent OOM errors
torch.cuda.empty_cache()
gc.collect()

print("✨ GPU Memory has been cleared.")
print("📁 You can now download your files from the 'outputs' folder on the left sidebar.")

# 🛠️ Troubleshooting Guide (UVR Edition)

If the separation process fails, check the solutions below:

### 1. ❌ "GPU not found" or NVIDIA errors
* **Cause:** Your Colab runtime is set to CPU or your GPU quota has ended.
* **Solution:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**. If it's already selected and still fails, you may need to wait 12-24h for a quota refresh from Google.

### 2. ❌ "FileNotFoundError: song.mp3 not found"
* **Cause:** The filename in Cell 4 does not match the file you uploaded.
* **Solution:** Check the `inputs` folder on the left sidebar. Ensure the filename has **no spaces** or special characters.
    * *Bad:* `My Song.mp3`
    * *Good:* `mysong.mp3`

### 3. ❌ "Out of Memory (OOM)"
* **Cause:** The GPU memory is full from a previous process.
* **Solution:** Run **Cell 5 (Cleanup)** to refresh the VRAM. If it persists, go to `Runtime` -> `Restart Session`.

### 4. ❌ "Model Download Failed"
* **Cause:** Connection issues with HuggingFace or Curl.
* **Solution:** Run **Cell 3** again. If it fails, manually check if you have enough disk space in the `/content` directory.

### 5. 🔊 Vocals still have background noise?
* **Tip:** Try a different model in **Cell 4**. `MDX_UVR_Vocal_FT` is usually better for songs with very loud instruments or backing vocals.